# Sprint A: Features, out-of-time split, EDA

**Order matters:** split *before* EDA, so every decision is made on training data only. Looking at the test set while choosing bands or features is a quiet form of leakage.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src.data_cleaning import load_and_clean
from src.features import add_features, feature_columns, time_split
from src.eda import default_rate_table, rank_variables, plot_default_rates

## 1. Feature engineering
Ratios usually capture risk better than raw amounts: a $500 payment means something very different on a $30k income vs a $150k income.

In [ ]:
df_model = load_and_clean("../data/loans_sample.csv.gz")
df_feat = add_features(df_model)

NEW = ["credit_hist_months", "fico", "pti", "dti_incl_loan", "loan_to_income", "revol_bal_to_income"]
df_feat[NEW].describe().round(3).T

## 2. Out-of-time split
- **Train:** Jan 2013 – Jun 2015
- **Test (OOT):** Jul 2015 – Dec 2015

Training starts in 2013 because most credit-bureau columns are empty before mid-2012.

In [ ]:
train, test = time_split(df_feat)

pd.DataFrame({
    "loans": [len(train), len(test)],
    "default_rate": [train["target"].mean(), test["target"].mean()],
    "from": [train["issue_d"].min().date(), test["issue_d"].min().date()],
    "to": [train["issue_d"].max().date(), test["issue_d"].max().date()],
}, index=["train", "test (OOT)"])

In [ ]:
FEATURES = feature_columns(train)
print(len(FEATURES), "features allowed into the model")

## 3. EDA on the training set

In [ ]:
EDA_COLS = [
    "fico", "dti_incl_loan", "revol_util", "inq_last_6mths", "annual_inc", "pti",
    "credit_hist_months", "emp_length", "mths_since_last_delinq",
    "purpose", "home_ownership", "verification_status",
]

rank_variables(train, EDA_COLS)

In [ ]:
plot_default_rates(train, EDA_COLS, save_path="../reports/figures/eda_default_rates.png")

In [ ]:
## zoom into one variable
default_rate_table(train, "emp_length")